# Altair Practice Lab

In this assignment we will be crafting a series of visualizations using
Altair to get practice working with real data in this context.

Your responses should be within the functions given,
using appropriate helper functions to help with clarity
and reduce redundancy.

## Rubric

The criteria to receive full credit on this assignment is a good-faith attempt at each portion.

A good-faith attempt should either:

- be fundamentally correct producing the expected output with minimal deviations,
- **OR** contain an explanation of what does not work and _details on what was tried_.

**For half credit, at least half of the assignment should have good-faith attempts.**

Your charts do not need to match the examples exactly!  They are helpful to get a sense of what you're after, but focus on the problem description and feel free to experiment as you see fit.

## Introducing the Dataset

In the data directory you'll find three files:

**legislators.csv** which consists of ~7400 records representing state legislators, it has the following fields:

- name
- given_name
- family_name
- party: As reported by the state.
- gender: Male / Female / Other\*
- jurisdiction: This field contains an identifier for the state or jurisdiction, see below for details.
- district: the name of the district represented
- type: upper | lower - The classification of the legislative. Most states have both, but DC and NE only have an upper chamber.

Note: Accurate data on gender is hard to come by in many states. There may be irregularities in this field. This is also why this field does not make further distinctions beyond Male/Female/Other.

**populations.csv** 

A CSV file with a row with each state's population.  

**actions.csv**

A CSV file with legislative actions for a few states across a range of years:

- description: action text as reported by state
- date: date of action recorded
- classification: not used for this assignment
- state: 2-letter state postal code (note: these are upper case, other files use lower case)
- session: an identifier representing which session the action comes from (not used)

In [ ]:
# PUT YOUR IMPORTS HERE
import pandas as pd
import altair as alt
import numpy as np

In [ ]:
# Disabling Altair's Max Rows limit
alt.data_transformers.disable_max_rows() 

## Part 1: Data Exploration

First, we'll build a few exploratory visualizations to get a sense of the data for this assignment.

### 1.0: Cleaning

As mentioned above, there is no 'state' field. Jurisdiction is in the format:

`ocd-jurisdiction/country:us/state:nc/government`

and for non-states:

- `ocd-jurisdiction/country:us/district:dc/government`
- `ocd-jurisdiction/country:us/territory:pr/government`

So for our purposes, we want to add a `state` column from the two letter code after either "state:", "district:", or "territory:".
(We will treat DC and PR as states.)

Complete the function `legislators_df` which should return the data from `legislators.csv` in a dataframe, with an additional `state` column.

In [ ]:
def legislators_df():
    df = pd.read_csv("data/legislators.csv")
    df['state'] = df['jurisdiction'].str[-13:-11]
    
    return df

# render dataframe
legislators_df()

### 1.1: Initial Plot

First let's build a visualization of gender breakdowns in state legislatures.
Use the following:

- stacked bars per state
- each segment of stacked bar is gender

Your graph should somewhat resemble *imgs/ex1.1.png*.

In [ ]:
def states_by_gender_initial(df):
    alt.data_transformers.disable_max_rows() 
    chart = alt.Chart(df).mark_bar().encode(
        alt.X('state').title("State"),
        alt.Y('count()').title("Count of Records"),
        alt.Color('gender:N').title("Gender")
    ).properties(
        title="Gender Breakdown of US State Legislatures"
    )
    return chart

# render chart
states_by_gender_initial(legislators_df())

### 1.2: Improvements

While it is clear from the first chart that there are more elected officials that are men than women, it is hard to compare across states.

Make the following adjustments:

- Normalize the chart so that each bar is a percentage, allowing for direct comparison across states.
- Since this is US political data, the colors red and blue have a strong meaning, associated with the Republican and Democratic parties. Change the color scheme to avoid red and blue. (I chose #8624f5 for women and #1fc3aa for men based on this article: <https://blog.datawrapper.de/gendercolor/>)
- Two states are very close to 50%, add a line at 50% using a layered chart to make it easier to see if they exceed 50% or not.

Your graph should somewhat resemble *imgs/ex1.2.png*.

In [ ]:
def states_by_gender_improved(df):
    
    alt.data_transformers.disable_max_rows() 
    
    genders = ["Female", "Male"]
    color_scheme = ["#8624f5", "#1fc3aa"]
    
    chart = alt.Chart(df).mark_bar().encode(
        alt.X('state:N').title("State"),
        alt.Y('count()', stack="normalize", axis=alt.Axis(format="%")).title("Count of Records"),
        alt.Color('gender:N').title("Gender").scale(
            domain=genders,
            range=color_scheme
        )
    ).properties(
        title="Gender Breakdown of US State Legislatures"
    )
    
    rule = alt.Chart().mark_rule().encode(y=alt.datum(0.5))
    return chart + rule

# render chart
states_by_gender_improved(legislators_df())

## Part 2: Party Breakdown

We'll now take a look at party control. We can start with essentially the same chart.

### 2.0 - Party Control

Copy your code from 1.2 above & modify it to use party instead of gender.  Your graph will wind up with too many parties, see `imgs/ex2.0.png`.

In [ ]:
def party_control_raw(df):
    chart = alt.Chart(df).mark_bar().encode(
        alt.X('state:N').title("State"),
        alt.Y('count()', stack="normalize", axis=alt.Axis(format="%")).title("Count of Records"),
        alt.Color('party:N').title("Political Party").scale(
        )
    ).properties(
        title="Party Breakdown of US State Legislatures"
    )
    
    rule = alt.Chart().mark_rule().encode(y=alt.datum(0.5))
    
    return chart + rule

party_control_raw(legislators_df())

### 2.1 - Cleaning Data

The above graph still has some shortcomings:

- Most states have an upper and lower chamber, and party control may vary between them. We'll need to make two bars per state (which we'll tackle in 2.2).
- Also, there are too many variations of party as you can see here:

Let's transform the data again, adding a new column "party_code" with the following rules:

- if the word 'Democratic' appears, set party_code to 'D'
- if the word 'Republican' appears, set the party_code to 'R'
- otherwise, set the party_code to 'O'

Party data in NE, DC, and PR does not work with this scheme.
For simplicity, we will exclude them from our analysis.

For this portion, implement `clean_party_df` which should return a modified legislators DataFrame with the `party_code` column, and the rows for states 'DC', 'NE' and 'PR' dropped.

In [ ]:
def clean_party_df():
    
    df = legislators_df()
    
    filter_list = ["DC", "NE", "PR"]
    df = df[~df['state'].isin(filter_list)]
    
    conditions = [
        df['party'].str.contains("Democratic", case=False, na=False),
        df['party'].str.contains("Republican", case=False, na=False)
    ]
    
    choices = ['D', 'R']
    
    df['party_code'] = np.select(conditions, choices, default="O")
    
    return df

clean_party_df()

### 2.2 - Faceted Plot

Add a function `party_control_by_chamber` that contains the following elements:

- One bar per state, **along the Y axis**.
- Each bar should consist of a stack: a blue portion, a green portion, and a red portion, corresponding to the D, O, and R `party_code` respectively.
- A vertical line at the 50% mark, indicating (likely) party control.
- Finally, facet the chart on `type` so that you get a set of bars for the lower and upper chambers.

See `imgs/ex2.2.png` for an example.

In [ ]:
def party_control_by_chamber(df):
    
    pol_party = ["D", "O", "R"]
    color_scheme = ["#0000FF","#00FF00","#FF0000"]
    
    chart = alt.Chart(df).mark_bar().encode(
        alt.X("count()", stack="normalize", axis=alt.Axis(format="%")
              ).title("Count of Records"),
        alt.Y("state:N").title("State"),
        alt.Color('party_code:N').title("Political Party").scale(
            domain=pol_party,
            range=color_scheme
        ),
    ).properties(
        title="Party Control by Chamber")
    
    rule = alt.Chart(df).mark_rule().encode(x=alt.datum(0.5))

    return alt.layer(chart, rule).facet("type")
  
party_control_by_chamber(clean_party_df())

## Part 3: Comparing by Population

For part three, we are interested in the relationship of various properties of legislatures to the total population of the state.

To do this, we'll need to create a combined DataFrame that mixes in data from `populations.csv`.

### 3.0 - Create Combined DataFrame

Write the function `population_combined_df`, which should return a DataFrame with the columns:

- state: abbreviation of state
- upper: total seats in upper chamber
- lower: total seats in lower chamber
- pop_2020: the 2020 population, obtained from merging with `population.csv`

**Data Note:** These numbers are based on the non-vacant seats as-of a particular day in September 2024. Vacancies will cause the counts to be off by a bit, but the general size of the legislature should be roughly the same.

In [ ]:
def population_combined_df():
   
    df = legislators_df()
    population_df = pd.read_csv("data/populations.csv")

    chamber_counts_df = pd.crosstab(df['state'], df['type']).reset_index()

    population_combined_df = pd.merge(chamber_counts_df[["state","upper","lower"]],population_df[["state","pop_2020"]], on="state", how="inner")

    return population_combined_df

population_combined_df()

### 3.1 - Create Population vs. Seats Scatterplot

Create a new plot with two layers:

- Population on the X axis
- Number of seats on the Y axis
- Upper chamber points should be purple and use the 'triangle-up' shape.
- Lower chamber points should be orange and use the 'triangle-down' shape.
- Make a customization or two to your chart's default labels and axes, whatever you feel is appropriate.

Hint: You can layer two charts for this.

In [ ]:
def scatter_pop_size():
    
    df = population_combined_df()
    
    chart = alt.Chart(df).transform_fold(
        ["upper", "lower"]
        ).mark_point().encode(
           x = alt.X('pop_2020:Q').title('Population in 2020'),
           y = alt.Y('value:Q').title('Number of Chamber Seats'),
           color= alt.Color(
               "key:N",
               scale = alt.Scale(
                   domain=["upper", "lower"],
                   range=["#8f34eb","#eb7734"])
            ),
           shape=alt.Shape(
               "key:N",
               scale = alt.Scale(
                   domain=["upper", "lower"],
                   range=["triangle-up","triangle-down"])
           )
        ).properties(title="2020 Population vs Chamber Seats")
    
    return chart

scatter_pop_size()

### 3.2 - Regressions

Add two more layers, a purple & orange regression line for each chamber.  See `imgs/ex3.2.png`

Hint: See `transform_regression`.

In [ ]:
def scatter_pop_size_regression():
    
    df = population_combined_df()
    
    chart = alt.Chart(df).transform_fold(
        ["upper", "lower"]
        ).mark_point().encode(
           x = alt.X('pop_2020:Q').title('Population in 2020'),
           y = alt.Y('value:Q').title('Number of Chamber Seats'),
           color= alt.Color(
               "key:N",
               scale = alt.Scale(
                   domain=["upper", "lower"],
                   range=["#8f34eb","#eb7734"])
            ),
           shape=alt.Shape(
               "key:N",
               scale = alt.Scale(
                   domain=["upper", "lower"],
                   range=["triangle-up","triangle-down"])
           )
        ).properties(title="2020 Population vs Chamber Seats")
    
    reg_lines = alt.Chart(df).transform_fold(
        ["upper", "lower"]
    ).transform_regression(
        on='pop_2020', 
        regression='value', 
        groupby=['key']
    ).mark_line().encode(
        x='pop_2020:Q',
        y='value:Q',
        color=alt.Color(
            "key:N",
            scale=alt.Scale(
                domain=["upper", "lower"],
                range=["#8f34eb", "#eb7734"])
        )
    )
    
    return chart + reg_lines

scatter_pop_size_regression()
    

## Part 4: Actions Heatmap

The file `actions_il-in-mi-wi_2021-2024.csv` contains nearly half a million records, representing every official action taken on a piece of legislation in these four midwestern states over the past two sessions.

Legislatures work quite differently, some meet all year, while others meet for very short periods.
By creating a heatmap of what days actions take place, we can get a sense of how different states compare.

### 4.0 - Load Actions

Complete `actions_df`, which should load the data from `actions_il-in-mi-wi_2021-2024.csv`.

Tips: 
- Make sure that the `date` column is loaded as a date type!
- Dates are in YYYY-MM-DD format, though some also have additional characters for time, which you will want to ignore.

In [ ]:
def actions_df():
    
    df = pd.read_csv("data/actions_il-in-mi-wi_2021-2024.csv")
    df['date'] = pd.to_datetime(df['date'], format='ISO8601', utc=True)
    
    return df

actions_df()

### 4.1 - Actions Heatmap

Generate a heatmap (using `mark_rect`) with:

- a row per state
- each row consists of shaded marks with shading based on the total action count for a given week

Tip: Use the 'yearweek(date)' aggregation for the X channel.

See `imgs/ex4.1.png`.

In [ ]:
def actions_heatmap_scaled(df):
    
    chart = alt.Chart(df).mark_rect().encode(
        x = ('yearweek(date):T'),
        y= alt.Y('state:N'),
        color = alt.Color("count():Q").title("actions")
    )
    return chart

actions_heatmap_scaled(actions_df())

### 4.2 - Excluding IL Outliers

Illinois clearly dominates the above graph, below modify two calls to `actions_heatmap` with a modified dataframe that excludes IL, and a modified dataframe that only includes IL.

(Note how by using functions in our dataframe we can more easily reuse portions by making small adjustments to the data.)

See `ex4.2a.png` and `ex4.2b.png`

In [ ]:
# MODIFY THIS LINE (exclude IL)
df = actions_df()
filtered_df = df[df['state'] != 'IL']

actions_heatmap_scaled(filtered_df)


In [ ]:
# MODIFY THIS LINE (only IL)
df = actions_df()
filtered_df = df[df['state'] == 'IL']

actions_heatmap_scaled(filtered_df)

#### 4.3 - Cumulative Line Chart

Another way to view this data would be with a cumulative line chart.

Create a chart with:

- days on the X axis
- cumulative actions to date on the Y axis
- one line per state

Hint: To do this you will need to look at the `transform_window` function.

See `ex4.3.png` for an example.

In [ ]:
def actions_cumulative(df):
    
    chart = alt.Chart(df).transform_aggregate(
        count="count()",
        groupby=["date","state"]
        ).transform_window(
        cumulative_count='sum(count)',
        groupby=['state'],
        sort=[alt.SortField('date', order='ascending')],
        frame=[None, 0]
    ).mark_line().encode(
        alt.X("yearmonthdate(date)"),
        alt.Y('cumulative_count:Q'),
        alt.Color('state')
    )

    return chart

actions_cumulative(actions_df())

## Part 5. Replicating a Chart

Find a chart (perhaps the one you're using for your first Analysis post!) and do your best to replicate it with Altair. Since you likely won't have access to the raw data-- start by mocking up a dataset that is similar enough to what your chart uses, don't worry about exact values, just that the correct attributes are present with some variation.

Your replicated chart should use similar colors, labels, etc. to the one that you chose to reproduce. Again, we aren't looking for a pixel-perfect replica, but something that pushes you to explore parts of Altair's documentation/features that you didn't use above.


![Alternative Text](DataViz_Analysis1.png)

In [ ]:
data = {
    "Food":['Lentils', 'Tomatoes', '2% Milk', 'Beans', 'Tofu', 'Broccoli', 'Yogurt',
            'Nuts', 'Peanut Butter', 'Rice', 'Potatoes', 'Eggs', 'Tuna', 'Chicken',
            'Turkey', 'Salmon', 'Pork', 'Cheese', 'Beef', 'Lamb'],
    "Miles Driven":[0.1, 0.2, 0.4, 0.4, 0.5, 0.5, 0.6, 0.7, 0.7, 0.7, 
                    0.8, 1.1, 1.5, 1.7, 2.7, 3.0, 3.0, 3.4, 6.6, 7.0]
}

df = pd.DataFrame(data).reset_index()


In [ ]:
chart = alt.Chart(df).mark_bar().encode(
    alt.X('Miles Driven:Q',axis=alt.Axis(orient='top')).title('Car Miles per 4oz. Consumed'),
    alt.Y('Food:N', sort=alt.EncodingSortField(
        field='index', order='ascending')),
    alt.Color('index:Q', legend=None).scale(scheme='redyellowgreen', reverse=True)
).properties(title="Carbon Footprint of Commonly Eaten Foods Represented in Equivalent Car Miles Driven")

chart.display()